# _common - shared helpers

Included by every pipeline notebook with `%run ./_common`. Defines:

* `CATALOG`, `JOB_RUN_ID`, `IN_JOB` (from widgets / job parameters)
* `table(schema, name)` - fully qualified, validated table names
* `add_missing_columns(table, {name: type})` - schema guard for tables created by an older `01_setup`
* `log_run(...)` - writes one row per task run to `ops.pipeline_runs`
* `task_run(task_name)` - context manager that times a task and logs SUCCESS / FAILED
* `stop_if_job()` - ends scheduled runs before the diagnostic "Inspect" cells (saves Free Edition compute)

In [ ]:
import json
import re
from contextlib import contextmanager
from datetime import datetime, timezone

dbutils.widgets.text("catalog", "jobseeker", "Catalog")
dbutils.widgets.text("job_run_id", "", "Job run id (set by the job)")

CATALOG = dbutils.widgets.get("catalog").strip()
if not re.fullmatch(r"[A-Za-z0-9_]+", CATALOG):
    raise ValueError(f"Invalid catalog name: {CATALOG!r}")

# The job passes job_run_id; interactive runs leave it empty
IN_JOB = bool(dbutils.widgets.get("job_run_id").strip())
JOB_RUN_ID = dbutils.widgets.get("job_run_id").strip() or f"manual-{datetime.now(timezone.utc):%Y%m%dT%H%M%S}"
# The run id is embedded in SQL text by several notebooks, so only allow a safe character set
if not re.fullmatch(r"[A-Za-z0-9_.:-]{1,64}", JOB_RUN_ID):
    raise ValueError(f"Invalid job_run_id: {JOB_RUN_ID!r}")
PIPELINE_VOLUME = f"/Volumes/{CATALOG}/ops/pipeline"


def table(schema, name):
    if not re.fullmatch(r"[A-Za-z0-9_]+", schema) or not re.fullmatch(r"[A-Za-z0-9_]+", name):
        raise ValueError(f"Invalid table identifier: {schema}.{name}")
    return f"{CATALOG}.{schema}.{name}"


def add_missing_columns(table_name, column_types):
    """Schema guard: adds any of {name: SQL type} that the table lacks (tables created by an older 01_setup)."""
    existing = {field.name for field in spark.table(table_name).schema.fields}
    missing = [f"{name} {kind}" for name, kind in column_types.items() if name not in existing]
    if missing:
        spark.sql(f"ALTER TABLE {table_name} ADD COLUMNS ({', '.join(missing)})")
        print(f"Added {missing} to {table_name}")
    return missing


def log_run(task, status, started_at, files_read=None, rows_in=None, rows_out=None, rows_rejected=None, details=None, message=None):
    row = [(
        JOB_RUN_ID, task, status, started_at, datetime.now(timezone.utc),
        files_read, rows_in, rows_out, rows_rejected,
        json.dumps(details or {}, default=str), (message or "")[:2000],
    )]
    schema = (
        "job_run_id STRING, task STRING, status STRING, started_at TIMESTAMP, finished_at TIMESTAMP, "
        "files_read INT, rows_in BIGINT, rows_out BIGINT, rows_rejected BIGINT, details STRING, message STRING"
    )
    spark.createDataFrame(row, schema).write.mode("append").saveAsTable(table("ops", "pipeline_runs"))


@contextmanager
def task_run(task):
    """Collects metrics in a dict and logs SUCCESS or FAILED to ops.pipeline_runs."""
    metrics = {"details": {}}
    started_at = datetime.now(timezone.utc)
    try:
        yield metrics
    except Exception as exc:
        log_run(task, "FAILED", started_at, message=f"{type(exc).__name__}: {exc}", details=metrics.get("details"))
        raise
    else:
        log_run(
            task, "SUCCESS", started_at,
            files_read=metrics.get("files_read"), rows_in=metrics.get("rows_in"), rows_out=metrics.get("rows_out"),
            rows_rejected=metrics.get("rows_rejected"), details=metrics.get("details"), message=metrics.get("message"),
        )
        print(f"[{task}] SUCCESS run={JOB_RUN_ID} {({k: v for k, v in metrics.items() if k != 'details'})}")


def stop_if_job():
    """Ends a scheduled job run before the 'Inspect' cells: they only display diagnostics, and every Spark action
    they trigger counts against the Free Edition compute quota. Interactive runs continue into them."""
    if IN_JOB:
        dbutils.notebook.exit("done")


print(f"catalog={CATALOG} job_run_id={JOB_RUN_ID} in_job={IN_JOB}")